# Exercise 7 : Combining Data

Compute the 2024 population of each World Bank income group and its share aged 65 and over. Count
each economy once, and keep every row that you leave out visible in an audit.

Run the first cell : it sets `DATA` to the course's `datasets` folder and `SOURCES` to the two
indicator files with their labels. Read `datasets/wb-population-README.md` first. The files are
unchanged World Bank downloads, retrieved on 10 October 2026 :

- `wb-wdi-population-total.csv` and `wb-wdi-population-65plus.csv` : World Development Indicators,
  with one row per economy or aggregate and one column per year
- `wb-class-economies.csv` : the income classification, one row per economy or aggregate
- `wb-class-groups.csv` : the group memberships, one row per economy and group

In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path.cwd().parents[1] / 'datasets'
SOURCES = {'total': 'wb-wdi-population-total.csv', '65plus': 'wb-wdi-population-65plus.csv'}

## Step 1 : Appending

Read the two indicator files into the list `frames`. Open one of them in a text editor first : four
metadata lines precede the header, so use `skiprows=4`. Every line ends with a comma, which creates
an empty column `Unnamed: 70`. Check that it is empty and drop it.

Add the labels of `SOURCES` as a `source` column and concatenate the frames once into
`indicators`, with a fresh integer index. Check that :

- `value_counts()` on `source` gives 266 rows per file, 532 in total
- No pair of `Country Code` and `Indicator Code` occurs twice
- Every country code is three capital letters : `str.fullmatch(r'[A-Z]{3}')`

Count the missing 2024 values as `missing_2024`. Which row is empty? Why is an empty value
different from zero people?

In [ ]:
frames = [pd.read_csv(DATA / name, skiprows=4) for name in SOURCES.values()]
print([frame['Unnamed: 70'].isna().all() for frame in frames])
frames = [frame.drop(columns='Unnamed: 70') for frame in frames]

indicators = pd.concat([frame.assign(source=label) for label, frame in zip(SOURCES, frames)],
                       ignore_index=True)
print(indicators['source'].value_counts())
print(indicators.duplicated(['Country Code', 'Indicator Code']).sum())
print(indicators['Country Code'].str.fullmatch(r'[A-Z]{3}').all())

missing_2024 = indicators['2024'].isna().sum()
print(missing_2024)
print(indicators.loc[indicators['2024'].isna(), ['Country Code', 'Country Name']])

## Step 2 : Duplicate keys

Keep the `Country Code`, `Indicator Code` and `2024` columns of `indicators` as `population_2024`.
Read `wb-class-groups.csv` into `groups`. Select every row whose `WB_Country_Code` occurs more than
once, keeping all occurrences, as `duplicate_groups`, and print the memberships of the Netherlands.

Try a left merge of `population_2024` with `groups` on `Country Code` and `WB_Country_Code`, with
`validate='many_to_one'`. Catch only `pd.errors.MergeError` and print its message. Why does a
membership list repeat every economy? Which of the two tables must have unique keys for this merge?

In [ ]:
population_2024 = indicators[['Country Code', 'Indicator Code', '2024']]
groups = pd.read_csv(DATA / 'wb-class-groups.csv')

duplicate_groups = groups.loc[groups.duplicated('WB_Country_Code', keep=False)]
print(len(duplicate_groups), 'of', len(groups))
print(groups.loc[groups['WB_Country_Code'] == 'NLD', ['WB_Group_Code', 'WB_Group_Name']])

try:
    population_2024.merge(groups, left_on='Country Code', right_on='WB_Country_Code',
                          how='left', validate='many_to_one')
except pd.errors.MergeError as error:
    print('Rejected:', error)

## Step 3 : Validated merge

Read `wb-class-economies.csv` into `economies`. Find and drop its one completely empty row with
`isna().all(axis=1)`, and check that `Code` is complete and unique.

Merge `population_2024` with `economies` into `combined`, with `how='left'`,
`validate='many_to_one'` and `indicator=True`. Check that `combined` has as many rows as
`population_2024` and the same 2024 total. Each country appears twice on the left, once per
indicator : why does the validation accept that?

In [ ]:
economies = pd.read_csv(DATA / 'wb-class-economies.csv')
blank = economies.isna().all(axis=1)
print(blank.sum())
economies = economies.loc[~blank]
print(economies['Code'].notna().all(), economies['Code'].is_unique)

combined = population_2024.merge(economies, left_on='Country Code', right_on='Code',
                                 how='left', validate='many_to_one', indicator=True)
print(len(combined), len(population_2024))
print(combined['2024'].sum() == population_2024['2024'].sum())

## Step 4 : Audit

Split `combined` into three parts and check that together they hold all its rows :

- `unmatched` : the `left_only` rows, whose code is not in `economies`
- `aggregates` : the matched rows without an income group, such as regions and the world
- `economies_2024` : the rows with an income group

Repeat the merge as `outer_audit`, with `how='outer'` and the same validation. Select its
`right_only` rows as `unused_economies` : economies without a row in the indicator files.

Compute three totals for 2024 :

- `world` : the total population of `WLD`
- `economy_total` : the total population summed over `economies_2024`
- `gap` : the difference between the two

Which economy explains the gap? Compare the High income aggregate, `HIC`, with the sum of its member
economies, then look at `unused_economies`. Explain in the answer cell whether leaving out the
aggregates and the unmatched rows loses population.

In [ ]:
unmatched = combined.loc[combined['_merge'] == 'left_only']
outer_audit = population_2024.merge(economies, left_on='Country Code', right_on='Code',
                                    how='outer', validate='many_to_one', indicator=True)
unused_economies = outer_audit.loc[outer_audit['_merge'] == 'right_only']
aggregates = combined.loc[(combined['_merge'] == 'both') & combined['Income group'].isna()]
economies_2024 = combined.loc[combined['Income group'].notna()]

print('Unmatched:', unmatched['Country Code'].unique().tolist())
print('Unused economies:', unused_economies['Code'].tolist())
print(len(unmatched) + len(aggregates) + len(economies_2024) == len(combined))

In [ ]:
totals = population_2024.loc[population_2024['Indicator Code'] == 'SP.POP.TOTL']
economy_totals = economies_2024.loc[economies_2024['Indicator Code'] == 'SP.POP.TOTL']
world = totals.loc[totals['Country Code'] == 'WLD', '2024'].iloc[0]
economy_total = economy_totals['2024'].sum()
gap = world - economy_total
print(f'World {world:,.0f} - economies {economy_total:,.0f} = {gap:,.0f}')

hic = totals.loc[totals['Country Code'] == 'HIC', '2024'].iloc[0]
high_income = economy_totals.loc[economy_totals['Income group'] == 'High income', '2024'].sum()
print(f'HIC {hic:,.0f} - members {high_income:,.0f} = {hic - high_income:,.0f}')
print(unused_economies[['Code', 'Economy', 'Income group']])

The gap is Taiwan, China (`TWN`) : the World Bank counts it in `WLD` and in `HIC`, but the WDI
files have no row for it, so the sum over the economies misses 23.4 million people. Leaving out
the aggregates and the unmatched rows loses no one : they repeat people who are already counted
in the economies.

## Step 5 : Income groups

Put the two indicators side by side per income group with `pivot_table(..., aggfunc='sum')` on
`economies_2024`, as `by_income`, and add the column `share_65plus`. Why is a sum safe here, while
the same call on a table that repeats an economy would hide a defect? The slide *Aggregating
Duplicate Pairs* shows the risk.

In [ ]:
by_income = economies_2024.pivot_table(index='Income group', columns='Indicator Code',
                                       values='2024', aggfunc='sum')
by_income['share_65plus'] = by_income['SP.POP.65UP.TO'] / by_income['SP.POP.TOTL']
print(by_income.round(3))

## If time permits :

1. Melt the year columns of the total-population file into `long`, keeping `Country Code` and
   `Country Name`, with the columns `year` and `population`. Predict the number of rows, then
   check it. Look up the Netherlands in 2024 by its labels : 17,993,485. Pivot `long` back into
   `restored` and look up the Netherlands in 2024 and in 1960 : 11,486,631.
2. The notebook builds `bad`, an unchecked left join of the 2024 population of the economies onto
   `groups`. Compute `original_total`, `bad_total` and `extra`. Count the rows per economy with
   `value_counts()` on `Country Code` as `repeats`, and find the smallest and largest count.
   Explain in two or three sentences in the answer cell which table repeats its keys, how often,
   why the total grows and what repairs it. Does `validate='many_to_many'` show that the result is
   correct?

In [ ]:
wide = frames[0]
years = [column for column in wide.columns if column.isdigit()]
long = wide.melt(id_vars=['Country Code', 'Country Name'], value_vars=years,
                 var_name='year', value_name='population')
print(len(long), len(wide) * len(years))
print(long.set_index(['Country Code', 'year']).loc[('NLD', '2024'), 'population'])

restored = long.pivot(index='Country Code', columns='year', values='population')
print(restored.loc['NLD', '2024'], restored.loc['NLD', '1960'])

In [ ]:
classified = economies.loc[economies['Income group'].notna(), 'Code']
totals_wide = frames[0]
population = totals_wide.loc[totals_wide['Country Code'].isin(classified),
                             ['Country Code', '2024']]
bad = population.merge(groups, left_on='Country Code', right_on='WB_Country_Code', how='left')

In [ ]:
original_total = population['2024'].sum()
bad_total = bad['2024'].sum()
extra = bad_total - original_total
print(f'{original_total:,.0f} {bad_total:,.0f} {extra:,.0f}')

repeats = bad['Country Code'].value_counts()
print(repeats.min(), repeats.max(), repeats['NLD'])

`groups` is a membership list : each economy appears once per group, between 3 and 17 times,
the Netherlands 7 times. The join repeats each population value as often, and the total grows
from 8.1 to 79.2 billion without one extra person. The economies lookup, merged with
`validate='many_to_one'`, repairs it. `validate='many_to_many'` accepts every result, so it
shows nothing about correctness.